In [ ]:

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk as nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import plotly.express as px
from collections import Counter
from nltk.tokenize import word_tokenize
import string
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc

Load Data

In [ ]:
data=pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
data.head(5)

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.sample()["clean_comment"].values

In [ ]:
data.isna().sum()

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.duplicated().sum()

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

In [ ]:
data[(data["clean_comment"].str.strip()=='')]

In [ ]:
data=data[~(data['clean_comment'].str.strip()=='')] #to remove \n

In [ ]:
#convert the clean-comment colums to lowercase
data["clean_comment"]=data["clean_comment"].str.lower()

data.head()

In [ ]:
# Remove trailing and leading whitespaces from the clean_comment colums
data["clean_comment"]= data["clean_comment"].str.strip()

In [ ]:
#identify comments containing URLs
url_patt = r'http[s]?://\S+'
comment_with_urls = data[data['clean_comment'].str.contains(url_patt, regex=True, na=False)]


In [ ]:
#remove \n from comment
data['clean_comment'] = data['clean_comment'].str.replace('\n', ' ', regex=True)


EDA


In [ ]:
# distribution of classes

sns.countplot(data=data,x="category")

In [ ]:
# frequency distribution of sentiments

data['category'].value_counts(normalize=True).mul(100).round(2)

In [ ]:
#to create a new colum word count
data["word_count"]=data["clean_comment"].apply(lambda x:len(x.split()))

In [ ]:
data.head()

In [ ]:
# distribution of word_count per sentiment

sns.displot(data,x='word_count',col='category',kind='kde')

In [ ]:
 #distribution of word_count per sentiment

sns.kdeplot(data,x='word_count',hue='category',fill=True)

In [ ]:
# plot wordclouds for each sentiment

def plot_wordcloud(target_class):
    text = ' '.join(data.loc[(data['category'] == target_class),"clean_comment"])
    wordcloud = WordCloud(width=800, height=600).generate(text)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Target-->{target_class}')
    plt.show()

for sentiment in data['category'].unique().tolist():
    plot_wordcloud(sentiment)

In [ ]:
# boxplots for pos tags for each sentiment


sns.boxplot(data["word_count"])

In [ ]:
data['msg_len'] = data['clean_comment'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10,5))
sns.histplot(data=data, x='msg_len', hue='category', bins=50, kde=True)
plt.title("Distribution of Message Length ( +ve , -ve , N)")
plt.xlabel("Message Length (words)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# avg word counts among sentiments

sns.barplot(data,x='category',y='word_count',estimator='mean')

In [ ]:
sns.barplot(data,x='category',y='word_count',estimator='median')

In [ ]:

# Download stopwords if not already
nltk.download("stopwords")

# English stopwords
stop_words = set(stopwords.words("english"))



# ---- Stopwords Count Column ----
def count_stopwords(text):
    words = re.findall(r'\w+', str(text).lower())
    return sum(word in stop_words for word in words)

data["num_stopwords"] = data["clean_comment"].apply(count_stopwords)

# kdeplot of stopwords count
sns.kdeplot(data=data, x='num_stopwords', hue='category', fill=True)
plt.title("Stopwords Count Distribution by Category")
plt.show()


In [ ]:

# kdeplot of stopwords count

sns.kdeplot(data,x='num_stopwords',hue='category',fill=True)


In [ ]:

# top 15 words and their count in each sentiment

def tokenize_sentences(text):
    tokens = nltk.word_tokenize(text)
    return tokens

tokens = data.loc[:,"clean_comment"].apply(tokenize_sentences)

positive_tokens = sum(tokens.loc[data['category'] == 1].tolist(), [])
neutral_tokens  = sum(tokens.loc[data['category'] == 0].tolist(), [])
negative_tokens = sum(tokens.loc[data['category'] == -1].tolist(), [])

In [ ]:
from collections import Counter


def get_top_n_words(tokens, n=20):
    return Counter(tokens).most_common(n)


top20_positive = get_top_n_words(positive_tokens, 20)
top20_neutral = get_top_n_words(neutral_tokens, 20)
top20_negative = get_top_n_words(negative_tokens, 20)

print("Top 20 Positive Words:")
print(top20_positive)
print("\n Top 20 Neutral Words:")
print(top20_neutral)
print("\n Top 20 Negative Words:")
print(top20_negative)


# ---- Visualization ----
def plot_top_words(top_words, sentiment_label):
    words, counts = zip(*top_words)
    plt.figure(figsize=(10,5))
    sns.barplot(x=list(counts), y=list(words), palette="viridis")
    plt.title(f"Top 20 Words - {sentiment_label}")
    plt.xlabel("Frequency")
    plt.ylabel("Words")
    plt.show()

plot_top_words(top20_positive, "Positive")
plot_top_words(top20_neutral, "Neutral")
plot_top_words(top20_negative, "Negative")

In [ ]:
import nltk, string, re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stopwords_En = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # lowercase
    text = text.lower()
    # remove punctuation
    text = "".join([char for char in text if char not in string.punctuation])
    # remove numbers
    text = re.sub(r'\d+', '', text)
    # tokenize
    tokens = word_tokenize(text)
    # remove stopwords
    tokens = [word for word in tokens if word not in stopwords_En]
    # lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

data['cleaned_text'] = data['clean_comment'].apply(preprocess_text)


In [ ]:
# ---- TF-IDF to DataFrame ----
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(2,2))
features_tfidf = tfidf.fit_transform(data['cleaned_text'])

#Sparse Matrix → DataFrame
sample_features = features_tfidf[:, :1000].toarray()

features_df = pd.DataFrame(sample_features, 
                           columns=tfidf.get_feature_names_out()[:1000])

print("Shape of TF-IDF DataFrame:", features_df.shape)
print(features_df.head())

# ---- Correlation Heatmap ----
sample_features = features_df.iloc[:, :30]  
corr = np.corrcoef(sample_features.T)

plt.figure(figsize=(12,8))
sns.heatmap(corr, cmap="coolwarm", 
            xticklabels=sample_features.columns, 
            yticklabels=sample_features.columns)
plt.title("Correlation Heatmap of Top 30 TF-IDF Features")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

tfidf = TfidfVectorizer(ngram_range=(1,3), max_features=5000)
X = tfidf.fit_transform(data['cleaned_text'])
y = data['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

# --- Classes ---
classes = [-1, 0, 1]
y_test_bin = label_binarize(y_test, classes=classes)
n_classes = y_test_bin.shape[1]

# --- Function to plot ROC ---
def plot_roc_curve(y_test_bin, y_score, model_name):
    plt.figure(figsize=(8,6))

    # لكل كلاس
    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f"Class {cls} (AUC = {roc_auc:.2f})")

    # Micro-average
    fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), y_score.ravel())
    roc_auc_micro = auc(fpr_micro, tpr_micro)
    plt.plot(fpr_micro, tpr_micro, linestyle=':', lw=3,
             label=f"Micro-average (AUC = {roc_auc_micro:.2f})")

    # Macro-average
    all_fpr = np.unique(np.concatenate([roc_curve(y_test_bin[:, i], y_score[:, i])[0] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        mean_tpr += np.interp(all_fpr, fpr, tpr)
    mean_tpr /= n_classes
    roc_auc_macro = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, linestyle='--', lw=3,
             label=f"Macro-average (AUC = {roc_auc_macro:.2f})")

    # Diagonal
    plt.plot([0, 1], [0, 1], 'k--', lw=2)

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {model_name}")
    plt.legend(loc="lower right")
    plt.show()

# --- 1. MultinomialNB ---
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_score_nb = nb.predict_proba(X_test)
plot_roc_curve(y_test_bin, y_score_nb, "MultinomialNB")

# --- 2. Logistic Regression ---
logreg = LogisticRegression(max_iter=1000, multi_class="ovr")
logreg.fit(X_train, y_train)
y_score_lr = logreg.predict_proba(X_test)
plot_roc_curve(y_test_bin, y_score_lr, "Logistic Regression")

# --- 3. SVC (decision_function) ---
svc = SVC(probability=False) 
svc.fit(X_train, y_train)
y_score_svc = svc.decision_function(X_test)

# decision_function (n_samples, n_classes)
plot_roc_curve(y_test_bin, y_score_svc, "SVC")


In [ ]:

# --- Classes ---
classes = [-1, 0, 1]
y_test_bin = label_binarize(y_test, classes=classes)
n_classes = y_test_bin.shape[1]

# --- Function For macro/micro AUC ---
def get_roc_auc(y_test_bin, y_score):
    fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), y_score.ravel())
    roc_auc_micro = auc(fpr_micro, tpr_micro)

    all_fpr = np.unique(np.concatenate(
        [roc_curve(y_test_bin[:, i], y_score[:, i])[0] for i in range(n_classes)]
    ))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        mean_tpr += np.interp(all_fpr, fpr, tpr)
    mean_tpr /= n_classes
    roc_auc_macro = auc(all_fpr, mean_tpr)

    return fpr_micro, tpr_micro, roc_auc_micro, all_fpr, mean_tpr, roc_auc_macro

# --- Train models ---
models = {
    "MultinomialNB": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class="ovr"),
    "SVC": SVC(probability=False)  
}

# --- Plot ---
plt.figure(figsize=(8,6))

for name, model in models.items():
    model.fit(X_train, y_train)

    if name == "SVC":
        y_score = model.decision_function(X_test)
    else:
        y_score = model.predict_proba(X_test)

    fpr_micro, tpr_micro, roc_auc_micro, all_fpr, mean_tpr, roc_auc_macro = get_roc_auc(y_test_bin, y_score)

    
    plt.plot(all_fpr, mean_tpr, lw=2, label=f"{name} (Macro AUC = {roc_auc_macro:.2f})")

# --- Diagonal line ---
plt.plot([0, 1], [0, 1], 'k--', lw=2)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison (Macro-average)")
plt.legend(loc="lower right")
plt.show()




3. متى تستخدم كل واحد؟

Macro → مهم لو عندك class imbalance (عايز تتأكد الموديل ما بيفضلش الكلاس الكبير على حساب الصغير).

Micro → مهم لو عايز تعرف الموديل ككل بيشتغل كويس ولا لأ، بصرف النظر عن توزيع الكلاسات.